# Token vs. Sentence Embeddings

## Antes de empezar: ¿qué vamos a comparar?

Este notebook compara dos formas de convertir texto en vectores (ver la sección "De palabras a oraciones: embeddings de token vs. de oración" de las notas de clase):

1. **Embeddings de token**, usando **BERT** (`bert-base-uncased`, vía la librería `transformers` de Hugging Face): BERT produce un vector por cada token (aproximadamente, cada palabra o subpalabra) de la entrada, capturando el significado de esa palabra **en su contexto** (a diferencia de un embedding estático tipo word2vec, la misma palabra tendrá vectores distintos según la oración en la que aparezca).

2. **Embeddings de oración**, usando un modelo tipo **SBERT/dual encoder** (`sentence-transformers`): en vez de un vector por token, obtenemos un único vector que resume el significado de toda la oración, entrenado específicamente para que la distancia entre dos embeddings de oración refleje qué tan similares son en significado.

No necesitas una cuenta ni una llave de API para nada de esto — ambos modelos son abiertos y se descargan la primera vez que los usas (por eso la primera ejecución de cada celda con `from_pretrained(...)` puede tardar unos segundos).

In [ ]:
#!pip install datasets

In [ ]:
# Dependencies
import torch
import numpy as np
import seaborn as sns
import pandas as pd
import scipy

from transformers import BertModel, BertTokenizer
from datasets import load_dataset

## Warning control
import warnings
warnings.filterwarnings('ignore')

## Setup

In [ ]:
#
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

In [ ]:
#
def get_sentence_embedding(sentence):
    encoded_input = tokenizer(sentence, padding=True, truncation=True, return_tensors='pt')
    attention_mask = encoded_input['attention_mask']   # to indicate which tokens are valid and which are padding

    # Get the model output (without the specific classification head)
    with torch.no_grad():
        output = model(**encoded_input)

    token_embeddings = output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # mean pooling operation, considering the BERT input_mask and padding
    sentence_embedding = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    return sentence_embedding.flatten().tolist()

In [ ]:
#
def cosine_similarity_matrix(features):
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    normalized_features = features / norms
    similarity_matrix = np.inner(normalized_features, normalized_features)
    rounded_similarity_matrix = np.round(similarity_matrix, 4)
    return rounded_similarity_matrix

In [ ]:
# Helper function to plot similarity matrix
def plot_similarity(labels, features, rotation):
    sim = cosine_similarity_matrix(features)
    sns.set_theme(font_scale=1.2)
    g = sns.heatmap(sim, xticklabels=labels, yticklabels=labels, vmin=0, vmax=1, cmap="YlOrRd")
    g.set_xticklabels(labels, rotation=rotation)
    g.set_title("Semantic Textual Similarity")
    return g

## Token Embeddings

In [ ]:
#
messages = [
    # Smartphones
    "I like my phone",
    "My phone is not good.",
    "Your cellphone looks great.",

    # Weather
    "Will it snow tomorrow?",
    "Recently a lot of hurricanes have hit the US",
    "Global warming is real",

    # Food and health
    "An apple a day, keeps the doctors away",
    "Eating strawberries is healthy",
    "Is paleo better than keto?",

    # Asking about age
    "How old are you?",
    "what is your age?",
]

In [ ]:
#
embeddings = []
for t in messages:
    emb = get_sentence_embedding(t)
    embeddings.append(emb)

plot_similarity(messages, embeddings, 90)

Acerca de las puntuaciones: Estas puntuaciones se basan en evaluaciones humanas y están en el rango de 0 a 5 (consulte la tabla 1 de https://aclanthology.org/S17-2001.pdf para ver cómo funcionan). Por ejemplo, "A woman is cutting onions" vs "A woman is cutting tofu" puede ser considerado por un humano como entre 2 y 3 aproximadamente. El objetivo de STS es más bien mostrar otro ejemplo en el que, en lugar de solo nuestra intuición, hay algún tipo de conjunto de datos etiquetados por humanos, aunque no es exactamente una similitud en el sentido del coseno, sino de acuerdo con las reglas de la tabla 1 del artículo referido.

In [ ]:
#
sts_dataset = load_dataset("mteb/stsbenchmark-sts")
sts = pd.DataFrame({'sent1': sts_dataset['test']['sentence1'],
                    'sent2': sts_dataset['test']['sentence2'],
                    'score': [x/5 for x in sts_dataset['test']['score']]})
sts.head(10)

In [ ]:
#
def sim_two_sentences(s1, s2):
    emb1 = get_sentence_embedding(s1)
    emb2 = get_sentence_embedding(s2)
    sim = cosine_similarity_matrix(np.vstack([emb1, emb2]))
    return sim[0,1]

n_examples = 50

sts = sts.head(n_examples)
sts['avg_bert_score'] = np.vectorize(sim_two_sentences) \
                                    (sts['sent1'], sts['sent2'])

In [ ]:
#
sts.head(10)

In [ ]:
#
pc = scipy.stats.pearsonr(sts['score'], sts['avg_bert_score'])
print(f'Pearson correlation coefficient = {pc[0]}\np-value = {pc[1]}')

## A better approach: SBERT and Dual Encoders

In [ ]:
#
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
#
embeddings = []
for t in messages:
    emb = list(model.encode(t))
    embeddings.append(emb)

plot_similarity(messages, embeddings, 90)

In [ ]:
#
def sim_two_sentences(s1, s2):
    emb1 = list(model.encode(s1))
    emb2 = list(model.encode(s2))
    sim = cosine_similarity_matrix(np.vstack([emb1, emb2]))
    return sim[0,1]

sts['mini_LM_score'] = np.vectorize(sim_two_sentences)(sts['sent1'], sts['sent2'])
sts.head(10)

In [ ]:
#
pc = scipy.stats.pearsonr(sts['score'], sts['mini_LM_score'])
print(f'Pearson correlation coefficient = {pc[0]}\np-value = {pc[1]}')